# 03-05-05 K-평균 실습 — 붓꽃과 원형군집

**(K-Means Practice with Iris & Blobs) - 이론이 처음으로 화면 위에서 움직이는 순간**

출처: 「Zero to 머신러닝 딥러닝 Master」 ([WikiDocs](https://wikidocs.net/book/21464)) — 이 노트북은 책의 코드 블록을 순서대로 담고 있습니다.


### Block 1 — 코드 — library

In [ ]:
# ── library ──────────────────────────────────────────────
from sklearn import datasets              # 연습용 표준 데이터 모음
import matplotlib.pyplot as plt          # 그림 그리기 (plt 는 관례적 별명)
import pandas as pd                       # 표 형태 데이터 다루기
from sklearn.cluster import KMeans         # 오늘의 주인공

# ── data ─────────────────────────────────────────────────
# 붓꽃(iris) : 꽃받침(sepal)과 꽃잎(petal)의 길이·너비 4개 특성으로
#              3가지 품종을 구분하는 고전 데이터셋 (150송이)
iris = datasets.load_iris()

# ── feature & target ─────────────────────────────────────
# iris.data 는 (150, 4) 배열입니다.
# [:, :2] 의 뜻 :
#     앞의  :   → "모든 행(150송이 전부)"
#     뒤의 :2   → "0번, 1번 열까지만" (2번은 미포함! 파이썬 슬라이싱 규칙)
# 즉 sepal length, sepal width 두 개만 골라 (150, 2) 로 만듭니다.
# 왜 2개만? → 2차원이어야 산점도로 눈에 보이기 때문입니다.
X = iris.data[:, :2]

# y 는 실제 품종 번호(0,1,2). ★ K-평균 학습에는 절대 쓰지 않습니다!
#    나중에 "채점"할 때만 몰래 꺼내 볼 용도입니다.
y = iris.target

print("X shape:", X.shape)
print("품종 이름:", iris.target_names)
print("사용한 특성:", iris.feature_names[:2])

# ── raw data 산점도 (정답 색으로 칠하기) ──────────────────
# c=y      : 점의 색을 y 값(0,1,2)에 따라 다르게
# cmap     : color map, 숫자를 색으로 바꾸는 규칙표
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='gist_rainbow')
plt.xlabel('Sepal Length', fontsize=18)
plt.ylabel('Sepal Width', fontsize=18)
plt.show()

### Block 2 — 코드 (강의 원본 → 최신 환경 수정본) — ✅ 최신 환경 수정본

In [ ]:
# ✅ 최신 환경 수정본
km = KMeans(
    n_clusters=3,        # K=3 : 품종이 3개인 걸 알고 있으니 3으로 (사람이 결정!)
    n_init=10,           # ★ 재시작 10회. n_jobs 대신 이것을 명시해야 합니다.
                          #   (최신 sklearn 기본값 'auto'는 1회라 결과가 달라집니다!)
    random_state=21       # 강의와 같은 시드 → 같은 결과 재현
)

km.fit(X)                 # ★ X만 넣습니다. y는 넣지 않습니다 (비지도학습!)

# cluster_centers_ : 학습이 끝난 뒤 만들어지는 (K, n) 배열
#                    언더스코어(_)로 끝나면 "학습 결과물"이라는 sklearn 규칙
centers = km.cluster_centers_
print(centers)
print("inertia (왜곡 합):", round(km.inertia_, 4))

### Block 3 — 코드 — labels_ : 각 데이터가 몇 번 군집인지 (150개짜리 정수 배열)

In [ ]:
# labels_ : 각 데이터가 몇 번 군집인지 (150개짜리 정수 배열)
new_labels = km.labels_

# ── 두 그림을 나란히 그려 비교 ─────────────────────────────
# subplots(1, 2) : 1행 2열로 그림판을 나눈다는 뜻.
#   fig  = 도화지 전체
#   axes = 각 칸의 좌표축 (axes[0], axes[1] 로 접근)
# figsize=(16,8) : 가로 16인치, 세로 8인치
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# 왼쪽 : 실제 품종(y)으로 색칠 — 정답지
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='gist_rainbow',
                edgecolor='k', s=150)   # edgecolor='k' 는 검은 테두리, s는 점 크기

# 오른쪽 : K-평균이 매긴 군집 번호로 색칠 — 예측
axes[1].scatter(X[:, 0], X[:, 1], c=new_labels, cmap='jet',
                edgecolor='k', s=150)

axes[0].set_xlabel('Sepal length', fontsize=18)
axes[0].set_ylabel('Sepal width', fontsize=18)
axes[1].set_xlabel('Sepal length', fontsize=18)
axes[1].set_ylabel('Sepal width', fontsize=18)

# tick_params : 눈금 모양 설정. direction='in' 은 눈금이 안쪽을 향하게
axes[0].tick_params(direction='in', length=10, width=5, colors='k', labelsize=20)
axes[1].tick_params(direction='in', length=10, width=5, colors='k', labelsize=20)

axes[0].set_title('Actual', fontsize=18)
axes[1].set_title('Predicted', fontsize=18)
plt.show()

### Block 4 — 🚨 가장 큰 함정 — "정확도"를 계산하려는 순간 — ✅ 올바른 방법 : 교차표(crosstab)로 대응 관계를 먼저 확인한다

In [ ]:
# ✅ 올바른 방법 : 교차표(crosstab)로 대응 관계를 먼저 확인한다
import pandas as pd

# pd.crosstab(행, 열) : 두 범주형 변수의 조합 개수를 세어 표로 만듭니다.
ct = pd.crosstab(y, km.labels_,
                 rownames=['실제 품종'],
                 colnames=['군집 번호'])
print(ct)

### Block 5 — 코드 — 데이터 생성

In [ ]:
from sklearn.datasets import make_blobs

# make_blobs : "덩어리(blob)를 만든다"는 뜻. 군집 실험용 인공 데이터 생성기.
X, y = make_blobs(
    n_samples=150,      # 점 150개
    n_features=2,       # 2차원 (그림으로 볼 수 있게)
    centers=3,          # 진짜 덩어리 3개 (우리는 알지만 모델은 모름)
    cluster_std=0.5,    # 덩어리의 퍼짐 정도. 작을수록 뭉쳐 있어 쉬움
    shuffle=True,       # 순서를 섞음 (앞부분만 보면 편향되지 않도록)
    random_state=0       # 시드 고정 → 몇 번을 돌려도 같은 데이터
)

# ── 정답을 숨긴 채 그려보기 ────────────────────────────────
# c='white' + edgecolor='black' : 속은 비고 테두리만 검은 동그라미
#   → "아직 어느 무리인지 모른다"는 상태를 표현하는 관례적 표현
plt.scatter(
    X[:, 0], X[:, 1],
    c='white', marker='o',
    edgecolor='black', s=50
)
plt.show()

### Block 6 — 코드 — 학습과 라벨 확인

In [ ]:
km = KMeans(
    n_clusters=3,
    init='random',     # 강의와 동일. (실무 권장은 'k-means++')
    n_init=10,          # 재시작 10회 → 지역 최적해 방어
    max_iter=300,       # 한 실행당 최대 300번 반복
    tol=1e-04,          # 중심 이동이 0.0001 미만이면 수렴으로 간주
    random_state=0
)

# fit_predict : fit()으로 학습하고 곧바로 labels_를 돌려주는 편의 함수.
#               km.fit(X); y_km = km.labels_  와 완전히 같습니다.
y_km = km.fit_predict(X)
print(y_km)

### Block 7 — 코드 — 군집별 색·모양과 중심점 별표 — X[y_km == 0, 0] 의 뜻을 천천히 뜯어봅시다.

In [ ]:
# X[y_km == 0, 0] 의 뜻을 천천히 뜯어봅시다.
#   y_km == 0      → [True, False, ...] 150개짜리 참/거짓 배열
#   X[그 배열, 0]  → True인 행만 골라서 0번 열(x좌표)만 꺼내기
# 즉 "0번 군집에 속한 점들의 x좌표"입니다. 뒤의 ,1 은 y좌표.
plt.scatter(
    X[y_km == 0, 0], X[y_km == 0, 1],
    s=50, c='lightgreen',
    marker='s', edgecolor='black',   # 's' = square(네모)
    label='cluster 1'
)
plt.scatter(
    X[y_km == 1, 0], X[y_km == 1, 1],
    s=50, c='orange',
    marker='o', edgecolor='black',   # 'o' = circle(동그라미)
    label='cluster 2'
)
plt.scatter(
    X[y_km == 2, 0], X[y_km == 2, 1],
    s=50, c='lightblue',
    marker='v', edgecolor='black',   # 'v' = 아래쪽 삼각형
    label='cluster 3'
)

# ── 중심점(centroids)을 큰 빨간 별로 ──────────────────────
plt.scatter(
    km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
    s=250, marker='*',            # 크게(250) 별표로 → 눈에 확 띄게
    c='red', edgecolor='black',
    label='centroids'
)

plt.legend(scatterpoints=1)   # 범례. scatterpoints=1은 견본 점을 1개만 표시
plt.grid()                     # 격자선
plt.show()

### Block 8 — 5단원 — K=4로 늘리면 생기는 일

In [ ]:
km = KMeans(n_clusters=4, init='random', n_init=10,
            max_iter=300, tol=1e-04, random_state=0)
y_km = km.fit_predict(X)

# 앞의 3개 군집 코드에 네 번째를 추가 (marker='d' = diamond, 마름모)
plt.scatter(X[y_km == 3, 0], X[y_km == 3, 1],
            s=50, c='lightblue', marker='d',
            edgecolor='black', label='cluster 4')

import numpy as np
print("각 군집 크기 :", np.bincount(y_km))
print("inertia      :", round(km.inertia_, 4))

### Block 9 — 6단원 — 엘보우 곡선으로 마무리

In [ ]:
distortions = []                     # 결과를 모을 빈 리스트

for i in range(1, 11):          # K = 1, 2, ..., 10
    km = KMeans(
        n_clusters=i, init='random',
        n_init=10, max_iter=300,
        tol=1e-04, random_state=0
    )
    km.fit(X)
    distortions.append(km.inertia_)  # 군집 내 분산, 적을수록 좋음

# plot - 급격하게 줄어드는 부분을 찾는다
plt.plot(range(1, 11), distortions, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Distortion')
plt.show()